In [ ]:
tuned_models = [
    "assets/MiniLM-triplet-m0.3/final",
    "assets/MiniLM-triplet-hyperbolic-m0.3-c0.3/final",
    "assets/MiniLM-triplet-dpo-m0.3-beta0.5/final",
]

org_model = "sentence-transformers/all-MiniLM-L6-v2"
output_prefix = "MiniLM"

# Taxonomy Learning

In [ ]:
from ontolearner import AutoRetrieverLearner
from ontolearner import evaluation_report
from ontolearner.utils import load_json, save_json
from tqdm import tqdm
import os
from ontolearner.ontology import FoodOn, GO, PO, SchemaOrg, OBI, SWEET

top_ks = [1, 5, 10]
task = 'taxonomy-discovery'

ontologies = [GO(),  SchemaOrg(), SWEET(), OBI(), PO()] # , ChEBI()

In [ ]:
logs = {}

for tuned_model in tuned_models:
    retriever_name = tuned_model.replace("/","_")
    retriever_id = tuned_model
    for top_k in top_ks:
        ret_learner = AutoRetrieverLearner(top_k=top_k, batch_size=5240)
        ret_learner.load(model_id=retriever_id)
        if top_k not in logs:
            logs[top_k] = {}
            
        for ontology in ontologies:
            if ontology.ontology_id not in logs[top_k]:
                logs[top_k][ontology.ontology_id] = {}
            ontology.load()
            data = ontology.extract()
            print("Working on :top-", top_k)
            ret_learner.fit(data, task=task)
            predicts = ret_learner.predict(data, task=task)
            truth = ret_learner.tasks_ground_truth_former(data=data, task=task)
            metrics = evaluation_report(y_true=truth, y_pred=predicts, task=task)
            logs[top_k][ontology.ontology_id][retriever_name] = metrics
            save_json(logs, f'{output_prefix}_taxonomy_learning.json')


In [ ]:
retriever_name = 'MiniLM-L6-v2'
retriever_id = org_model

for top_k in top_ks:
    ret_learner = AutoRetrieverLearner(top_k=top_k, batch_size=5240)
    ret_learner.load(model_id=retriever_id)
    for ontology in ontologies:
        ontology.load()
        data = ontology.extract()
        print("Working on :top-", top_k)
        ret_learner.fit(data, task=task)
        predicts = ret_learner.predict(data, task=task)
        truth = ret_learner.tasks_ground_truth_former(data=data, task=task)
        metrics = evaluation_report(y_true=truth, y_pred=predicts, task=task)
        logs[top_k][ontology.ontology_id][retriever_name] = metrics
         save_json(logs, f'{output_prefix}_taxonomy_learning.json')

# OA

In [ ]:
import json
from ontoaligner.ontology import GenericOMDataset

from ontoaligner.utils import metrics, xmlify
from ontoaligner.aligner import SBERTRetrieval
from ontoaligner.postprocess import retriever_postprocessor
from ontoaligner.encoder import ConceptLightweightEncoder

ontologies_dirs = ['oa/envo-sweet', 'oa/mouse-human', 'oa/MaterialInformation-MatOnto', 'oa/yago-wikidata']

def run_alignment(source_ontology_path, target_ontology_path, reference_matching_path, aligner):
    task  = GenericOMDataset()
    dataset = task.collect(
        source_ontology_path=source_ontology_path,
        target_ontology_path=target_ontology_path,
        reference_matching_path=reference_matching_path
    )
    encoder = ConceptLightweightEncoder()
    encoder_output = encoder(source=dataset['source'], target=dataset['target'])
    matchings = aligner.generate(input_data=encoder_output)
    matchings = retriever_postprocessor(matchings)
    evaluation = metrics.evaluation_report(
        predicts=matchings,
        references=dataset['reference']
    )
    return evaluation 


In [ ]:
top_ks = [1, 5, 10]

eval_logs = {}
org_model_name = 'MiniLM-L6-v2'
for top_k in top_ks:
    aligner = SBERTRetrieval(device='cuda', top_k=top_k)
    aligner.load(path=org_model)
    if top_k not in eval_logs:
        eval_logs[top_k] = {}
        
    for ontology_dir in ontologies_dirs:    
        
        print("working on ontology_dir:", ontology_dir)
        task = ontology_dir.split("/")[-1]
        if task not in eval_logs[top_k]:
            eval_logs[top_k][task] = {}
        evaluation  = run_alignment(source_ontology_path=f'{ontology_dir}/source.xml', 
                                    target_ontology_path=f'{ontology_dir}/target.xml', 
                                    reference_matching_path = f'{ontology_dir}/reference.xml',
                                    aligner=aligner)
        eval_logs[top_k][task][org_model_name] = evaluation
        save_json(eval_logs, f'{output_prefix}_ontology_alignment.json')
        


In [ ]:
for top_k in top_ks:
    for tuned_model in tuned_models:
        model_name = tuned_model.replace("/","_")
        aligner = SBERTRetrieval(device='cuda', top_k=top_k)
        aligner.load(path=tuned_model)
        
        for ontology_dir in ontologies_dirs:    

            print("working on ontology_dir:", ontology_dir)
            task = ontology_dir.split("/")[-1]
            evaluation  = run_alignment(source_ontology_path=f'{ontology_dir}/source.xml', 
                                        target_ontology_path=f'{ontology_dir}/target.xml', 
                                        reference_matching_path = f'{ontology_dir}/reference.xml',
                                        aligner=aligner)
            eval_logs[top_k][task][model_name] = evaluation
            save_json(eval_logs, f'{output_prefix}_ontology_alignment.json')

